# RF Complexity Analysis (EU)
## Data Loading

We use the `forest_report.json` file in the repo and the unified ETL cache system.

In [ ]:
from pathlib import Path
from etl.loader import etl

RESULTS_DIR = Path("results")
zip_paths = sorted(RESULTS_DIR.glob("*.zip"))

db = etl(
    zip_paths,
    RESULTS_DIR,
    use_cache=True,           # Use the unified cache system
    force_refresh=False,      # Set to True to force refresh
    auto_select=True,
    load_only_db10=True,      # Load ONLY DB10
    verbose=False
)

from etl.tables import (
    prepare_models_analysis,
    print_models_analysis_diagnostics,
)

analysis_context = prepare_models_analysis(db=db, verbose=True, selected_dataset=None)

In [ ]:
analysis_context.first_table.summary_styler

In [ ]:
print_models_analysis_diagnostics(analysis_context)
analysis_context.summary_styler

In [ ]:
# Display the combined analyzed table
print("📊 Combined Analyzed Results (Test Accuracy & Performance Metrics):")
print("="*80)
analysis_context.combined_analyzed_styler

In [ ]:
# Save the combined analyzed data to CSV
combined_df = analysis_context.combined_analyzed_styler.data
combined_df.to_csv("combined_analyzed_results.csv")

print(f"✅ Saved combined analysis results to: combined_analyzed_results.csv")
print(f"📊 Shape: {combined_df.shape}")
print(f"🔍 Columns: {list(combined_df.columns)}")
print(f"📋 Index: {list(combined_df.index)}")

# Also save the first table summary
summary_df = analysis_context.first_table.summary_styler.data
summary_df.to_csv("summary_results.csv", index=False)

print(f"✅ Saved summary results to: summary_results.csv")
print(f"📊 Shape: {summary_df.shape}")
print(f"🔍 Columns: {list(summary_df.columns)}")

# Save the analyzed counts
counts_df = analysis_context.analyzed_counts_df
counts_df.to_csv("analyzed_counts.csv", index=False)

print(f"✅ Saved analyzed counts to: analyzed_counts.csv") 
print(f"📊 Shape: {counts_df.shape}")
print(f"🔍 Columns: {list(counts_df.columns)}")

In [ ]:
import pandas as pd
combined_df = pd.read_csv("combined_analyzed_results.csv")

In [ ]:
combined_df

In [ ]:
# Combine Mean EU Features and EU Std into a single row, then add data from other CSVs
import pandas as pd
import numpy as np

# Read the current combined data
combined_df = pd.read_csv("combined_analyzed_results.csv", index_col=0)

# Check if both rows exist
if 'Mean EU Features' in combined_df.index and 'EU Std' in combined_df.index:
    # Get the data for both rows
    mean_row = combined_df.loc['Mean EU Features']
    std_row = combined_df.loc['EU Std']
    
    # Create the combined row with format "mean (± std)"
    combined_row = pd.Series(index=combined_df.columns, dtype=object)
    
    for col in combined_df.columns:
        mean_val = mean_row[col]
        std_val = std_row[col]
        
        # Check if both values are valid numbers
        if pd.notna(mean_val) and pd.notna(std_val):
            try:
                mean_num = float(mean_val)
                std_num = float(std_val)
                combined_row[col] = f"{mean_num:.3f} (± {std_num:.3f})"
            except (ValueError, TypeError):
                combined_row[col] = str(mean_val)  # fallback to original mean value
        elif pd.notna(mean_val):
            combined_row[col] = str(mean_val)
        else:
            combined_row[col] = ""
    
    # Remove the original rows and add the combined row
    combined_df = combined_df.drop(['Mean EU Features', 'EU Std'])
    
    # Insert the combined row at the position where Mean EU Features was
    # Find the best position (after N Estimators, before Test Accuracy if it exists)
    insert_idx = 4  # Default position
    if 'N Estimators' in combined_df.index:
        insert_idx = list(combined_df.index).index('N Estimators') + 1
    
    # Insert the new row
    combined_df_list = combined_df.index.tolist()
    combined_df_list.insert(insert_idx, 'Mean EU Features (± Std)')
    
    # Reindex with the new order
    new_combined_df = pd.DataFrame(index=combined_df_list, columns=combined_df.columns)
    for idx in combined_df.index:
        new_combined_df.loc[idx] = combined_df.loc[idx]
    new_combined_df.loc['Mean EU Features (± Std)'] = combined_row
    
    combined_df = new_combined_df
    
    print("✅ Successfully combined Mean EU Features and EU Std")

# Now add data from summary_results.csv and analyzed_counts.csv
try:
    # Load summary results
    summary_df = pd.read_csv("summary_results.csv")
    print(f"✅ Loaded summary_results.csv with {summary_df.shape[0]} datasets")
    
    # Load analyzed counts
    counts_df = pd.read_csv("analyzed_counts.csv")
    print(f"✅ Loaded analyzed_counts.csv with {counts_df.shape[0]} datasets")
    
    # Create additional rows for summary data that's not already in combined_df
    summary_metrics_to_add = ['n_features','eu_complexity','eu_min','eu_max']
    
    
    # Add summary metrics as new rows
    for metric in summary_metrics_to_add:
        if metric in summary_df.columns:
            new_row = pd.Series(index=combined_df.columns, dtype=object)
            
            # Map dataset names to values
            for _, row in summary_df.iterrows():
                dataset = str(row['dataset'])
                if dataset in combined_df.columns:
                    value = row[metric]
                    if pd.notna(value):
                        new_row[dataset] = value
            
            # Add the new row
            metric_name = metric.replace('_', ' ').title()
            if metric == 'n_features':
                metric_name = 'N Features'
            elif metric == 'eu_complexity':
                metric_name = 'EU Complexity'
            elif metric == 'eu_min':
                metric_name = 'EU Min'
            elif metric == 'eu_max':
                metric_name = 'EU Max'
                
            combined_df.loc[metric_name] = new_row
            print(f"✅ Added {metric_name} from summary_results.csv")
    
    # Add metrics from analyzed_counts that aren't already present
    counts_metrics_to_add = [
        'selected_sample',  # If this column exists
    ]
    
    # Check what additional metrics are in counts_df
    for col in counts_df.columns:
        if col != 'dataset' and col not in ['Total time (s) max', 'Total time (s) mean', 
                                           'ICF checks', 'Reason check iteration total',
                                           'IterGoodRatio', 'IterBadRatio', 'Early Stop Good total',
                                           'Early Stop from Good', 'Early Stop from Bad', 'Filtrered rate']:
            counts_metrics_to_add.append(col)
    
    # Add counts metrics as new rows
    for metric in counts_metrics_to_add:
        if metric in counts_df.columns:
            new_row = pd.Series(index=combined_df.columns, dtype=object)
            
            # Map dataset names to values
            for _, row in counts_df.iterrows():
                dataset = str(row['dataset'])
                if dataset in combined_df.columns and dataset != 'All workers':
                    value = row[metric]
                    if pd.notna(value):
                        new_row[dataset] = value
            
            # Add the new row if it has any data
            if new_row.notna().any():
                metric_name = metric.replace('_', ' ').title()
                combined_df.loc[metric_name] = new_row
                print(f"✅ Added {metric_name} from analyzed_counts.csv")
    
    print(f"\n📊 Final combined dataframe shape: {combined_df.shape}")
    print(f"📋 Total metrics: {len(combined_df.index)}")
    print(f"🏷️  Datasets: {len(combined_df.columns)}")
    
except Exception as e:
    print(f"❌ Error adding data from CSV files: {e}")

# Display the updated dataframe
combined_df.to_csv("combined_analyzed_results_updated.csv")
combined_df